# Setup

In [57]:
import pandas as pd
from pathlib import Path


# Get the current directory
current_directory = Path.cwd()

# Get results folder
dir_dat = current_directory.parent.parent.parent / "6_scRNAseq organoid data"

# Read in data from meet in middle analysis
df = pd.read_csv(dir_dat / 'Excel Supplement B genes brittneys paper.csv')

# Get names of genes
genes = df.gene

# Split symbols on '|', and flatten the list
split_genes = [symbol for item in genes for symbol in item.split('; ')] 

# PNPLA3-I148M eQTLs:
PNPLA3_eqtls = ['PNPLA5', 'EFCAB6', 'RTL6', 'SULT4A1', 'PARVG', 'SAMM50', 'MPPED1', 'PNPLA3', 'PARVB', 'SHISAL1']

# Add PNPLA3
split_genes.extend(PNPLA3_eqtls)

# Convert to list of strings in the format you want
gene_list = "[" + ", ".join(["'" + symbol + "'" for symbol in split_genes]) + "]"
gene_list = gene_list.replace(' ', '')

print(gene_list)

['APOA4','PLIN2','HMGCS2','FABP1','HADHB','PDK4','ACSL1','HADHA','PCK1','APOA2','CYP3A4','MT1F','FBP1','RPL23','ACAA1','RPS20','ACAA2','ANGPTL4','ACADVL','ABCB4','MT1X','CYP3A7','G6PC','ACADM','CYP4A11','ACOX1','APOA5','SLC25A47','CYP2B6','SERPINE1','AKR1C3','SAA4','AGT','ACTG1','TPM1','TM4SF4','TUBB2A','LINC01435','PKN2-AS1','APOH','CYP2C9','UGT2B10','PTGR1','UGT2B7','FN1','ADH1A','ANG','ANGPTL8','TIMP1','AHSG','SERPINA3','UGT2B15','ADH1C','GSTA1','UGT2B4','AKR1B10','FGA','S100A6','FGG','FGB','PNPLA5','EFCAB6','RTL6','SULT4A1','PARVG','SAMM50','MPPED1','PNPLA3','PARVB','SHISAL1']


In [2]:
! python --version

Python 3.10.9


# 1. Query ComptoxAI database using Neo4j.
## 1.a. Identify all nodes linking PFNA, genes, and the outcome. 
Note: to get this to run, you have to have an Neo4j instance of comptoxai.

In [12]:
import comptox_ai
from comptox_ai.db.graph_db import GraphDB
db = GraphDB(username="cytoscape", password="12345", hostname="localhost:7687")

# Define Disease and Exposure Names for ComptoxAI 
chemical_common_name = 'Perfluoroheptanoic acid'
disease_common_name = 'Liver carcinoma'

# Create cypher query
# Start and end strings
start_string = "MATCH (d:Disease {commonName: 'Liver carcinoma'}) MATCH (end:Gene) WHERE end.geneSymbol IN "

middle_string = " CALL apoc.path.spanningTree(d, {relationshipFilter: '<GENEASSOCIATESWITHDISEASE|GENEINTERACTSWITHGENE', minLevel: 1, maxLevel: 3, endNodes: end}) YIELD path WITH nodes(path) as n RETURN n UNION MATCH (c:Chemical {commonName: 'Perfluoroheptanoic acid'}) MATCH (d:Disease {commonName: 'Liver carcinoma'}) MATCH (end:Gene) WHERE end.geneSymbol IN"

end_string = " CALL apoc.path.spanningTree(c, {relationshipFilter: 'CHEMICALINCREASESEXPRESSION>|CHEMICALDECREASESEXPRESSION>|GENEINTERACTSWITHGENE>', minLevel: 1, maxLevel: 3, endNodes: end}) YIELD path WITH nodes(path) as n RETURN n;"

# Combine the start and end strings with the unique_gene_symbols_list
first_query = start_string + gene_list + middle_string + gene_list + end_string
#print(first_query)

# Run cypher query
nodes = db.run_cypher(first_query) 

## 1.b. Clean the gene names from the first query

In [13]:
# Unlist everything and get a list of separate dictionaries
flattened_list = [inner_dict for outer_dict in nodes for inner_dict in outer_dict['n']]

# Remove duplicates by converting to a set and back to a list
unique_list = list({tuple(item.items()) for item in flattened_list})

# Convert the list of tuples back to dictionaries
unique_list = [dict(item) for item in unique_list]

print('The number of unique nodes, including the PFAS and disease node, is ' +  str(len(unique_list)))
# Look at first few elements of list
#print(unique_list[0:2])

# Extract geneSymbol
# Create a set to store unique geneSymbol values
unique_gene_symbols = set()

# Iterate through the list and collect unique geneSymbol values
for dictionary in flattened_list:
    if 'geneSymbol' in dictionary:
        unique_gene_symbols.add(dictionary['geneSymbol'])

# Convert the set back to a list if needed
unique_gene_symbols_list = list(unique_gene_symbols)

# Now, 'unique_gene_symbols_list' contains all unique geneSymbol values

print('The number of unique genes is ' +  str(len(unique_gene_symbols_list)) + ", and " + str(len(set(unique_gene_symbols_list) & set(split_genes))) + " of these genes are from the original data.")

#print(unique_gene_symbols_list)

The number of unique nodes, including the PFAS and disease node, is 115
The number of unique genes is 113, and 63 of these genes are from the original data.


## 1.c. With all genes, identify all pathways linked to at least 10 genes

In [15]:
start_string = "MATCH (gene:Gene)-[:GENEINPATHWAY]->(pw) WHERE gene.geneSymbol IN"

end_string = " WITH pw, COUNT(DISTINCT gene) AS geneCount WHERE geneCount >= 10 RETURN pw;"

# Combine the start and end strings with the unique_gene_symbols_list
second_query = start_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_gene_symbols_list]) + "]" + end_string
#print(second_query)

# Run query
pathway_nodes = db.run_cypher(second_query) 

#### Clean Pathway Node Names

In [16]:
# Unlist everything and get a list of separate dictionaries
flattened_list_pw = [outer_dict['pw'] for outer_dict in pathway_nodes if 'pw' in outer_dict]


# Remove duplicates by converting to a set and back to a list
unique_list_pw = list({tuple(item.items()) for item in flattened_list_pw})

# Convert the list of tuples back to dictionaries
unique_list_pw = [dict(item) for item in unique_list_pw]

# Look at first few elements of list
#print(unique_list_pw[0:4])

# Extract pathwayId
# Create a set to store unique geneSymbol values
unique_pw_id = set()

# Iterate through the list and collect unique geneSymbol values
for dictionary in unique_list_pw:
    if 'commonName' in dictionary:
        unique_pw_id.add(dictionary['commonName'])

# Convert the set back to a list if needed
unique_pw_id_list = list(unique_pw_id)

# Now, 'unique_pw_id_list' contains all unique geneSymbol values

print('The number of unique pathways is ' +  str(len(unique_pw_id_list)))
print(unique_pw_id_list)

The number of unique pathways is 26
['Immune System', 'Pathways in cancer - Homo sapiens (human)', 'Signal Transduction', 'Signaling by GPCR', 'Metabolism of lipids and lipoproteins', 'Hemostasis', 'Tyrosine metabolism', 'Fatty acid degradation - Homo sapiens (human)', 'Extracellular matrix organization', 'Axon guidance', 'PPAR signaling pathway - Homo sapiens (human)', 'Regulation of lipid metabolism by Peroxisome proliferator-activated receptor alpha (PPARalpha)', 'Regulation of Insulin-like Growth Factor (IGF) transport and uptake by Insulin-like Growth Factor Binding Proteins (IGFBPs)', 'Proteoglycans in cancer - Homo sapiens (human)', 'Platelet activation, signaling and aggregation', 'Metapathway biotransformation', 'Metabolic pathways', 'Nuclear Receptors Meta-Pathway', 'Leukotriene metabolism', 'PPARA activates gene expression', 'Innate Immune System', 'Developmental Biology', 'Platelet degranulation ', 'Metabolism', 'Biological oxidations', 'Aminosugars metabolism']


## 1.d. With all PFAS, Genes, pathways, and disease, identify all of the relationships

In [25]:
# Create cypher query
# Start and end strings
start_string = "MATCH (c:Chemical {commonName: 'Perfluoroheptanoic acid'}) MATCH (d:Disease {commonName: 'Liver carcinoma'}) MATCH (node1:Gene) WHERE node1.geneSymbol IN "

middle_string = "MATCH (node2:Pathway) WHERE node2.commonName IN "

end_string = "WITH collect(id(node1))+collect(c)+collect(d)+collect(id(node2)) as nodes CALL apoc.algo.cover(nodes) YIELD rel RETURN  startNode(rel), rel, endNode(rel);"

# Combine the start and end strings with the unique_gene_symbols_list
third_query = start_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_gene_symbols_list]) + "]" + middle_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_pw_id_list]) + "]" + end_string

# Run Cypher Query
data = db.run_cypher(third_query)

# 2. Create Graph

In [43]:
# Create network diagram
import networkx as nx
import matplotlib.pyplot as plt
# Create a new graph
G = nx.DiGraph()

# List mediator genes (defined above):
# PNPLA3_eqtls

# List mediator genes
organoid_snRNAseq_genes = split_genes

# Function to compute the combined 'type' attribute
def compute_type(node):
    if node.get('commonName') == 'Perfluoroheptanoic acid':
        return 'PFAS'
    if node.get('') == '': 
        return 'Pathway'
    if node.get('commonName') == 'Diabetic Nephropathy':
        return 'Disease'
    if node.get('geneSymbol') in organoid_snRNAseq_genes:
            return 'in_vitro_gene'
    else:
        if node.get('geneSymbol') in PNPLA3_eqtls:
            return 'PNPLA3-I148M eQTL'
        else:
            return 'Other gene'


In [56]:
print(data[1])
#data[1]
'pathwayId' in data[4]

{'startNode(rel)': {'typeOfGene': 'protein-coding', 'commonName': 'thymic stromal lymphopoietin', 'xrefOMIM': '607003', 'xrefHGNC': '30743', 'xrefEnsembl': 'ENSG00000145777', 'geneSymbol': 'TSLP', 'uri': 'http://jdr.bio/ontologies/comptox.owl#gene_tslp', 'xrefNcbiGene': 85480, 'type': 'Other gene'}, 'rel': ({'typeOfGene': 'protein-coding', 'commonName': 'thymic stromal lymphopoietin', 'xrefOMIM': '607003', 'xrefHGNC': '30743', 'xrefEnsembl': 'ENSG00000145777', 'geneSymbol': 'TSLP', 'uri': 'http://jdr.bio/ontologies/comptox.owl#gene_tslp', 'xrefNcbiGene': 85480}, 'GENEINPATHWAY', {'commonName': 'Aminosugars metabolism', 'sourceDatabase': 'INOH', 'uri': 'http://jdr.bio/ontologies/comptox.owl#pathway_none', 'pathwayId': 'None'}), 'endNode(rel)': {'commonName': 'Aminosugars metabolism', 'sourceDatabase': 'INOH', 'uri': 'http://jdr.bio/ontologies/comptox.owl#pathway_none', 'pathwayId': 'None', 'type': 'Other gene'}}


False

In [45]:
# Add nodes and edges with combined 'type' attribute
for entry in data:
    start_node = entry['startNode(rel)']
    end_node = entry['endNode(rel)']
    rel_type = entry['rel'][1]  # Relationship type is the second item in the tuple
    
    # Set combined 'type' attribute
    start_node['type'] = compute_type(start_node)
    end_node['type'] = compute_type(end_node)

    # Node identifiers
    start_node_id = start_node.get('geneSymbol')  or start_node.get('commonName')
    end_node_id = end_node.get('geneSymbol') or end_node.get('xrefUmlsCUI') 

    # Add nodes with combined 'type' attribute
    G.add_node(start_node_id, **start_node)
    G.add_node(end_node_id, **end_node)

    # Add edge
    G.add_edge(start_node_id, end_node_id, relationship=rel_type)

ValueError: None cannot be a node

In [ ]:
# Color mapping
color_map = {
    'PFAS': 'magenta',
    'Disease': 'red',
    'mediating-protein': 'green',
    'non-identified protein': 'blue'
}

# Compute node colors based on 'type' attribute
node_colors = [color_map[G.nodes[node]['type']] for node in G]

# Draw the graph
plt.figure(figsize=(12, 8))  # Set the figure size
pos = nx.spring_layout(G)  # Layout for the nodes
nx.draw_networkx(G, pos, with_labels=True, node_color=node_colors, node_size=700, edge_color='k', linewidths=1, font_size=10, arrows=True)
#nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'relationship'))

plt.show()

In [ ]:
# Save file

file_path = dir_res / 'ComptoxAI' / 'PFHpA_HCC_expanded_020924.graphml'

# Write the graph to a GraphML file
nx.write_graphml(G, file_path)

print(f"Network saved to {file_path}")